In [1]:
from datetime import datetime, timedelta, timezone
from sqlalchemy import text

%run _1_schema.ipynb
%run _6_config_intervals.ipynb

# Archive job:
# 1) Move old intraday candles from `stock_ohlc` to `stock_intraday_archive`
# 2) Delete moved rows from `stock_ohlc`
# 3) Cleanup very old rows in `stock_intraday_archive`


def _ensure_tables():
    engine = get_engine()
    get_stock_ohlc_table(engine)
    get_intraday_archive_table(engine)
    return engine


def archive_old_intraday(batch_limit: int | None = None, sleep_sec: float = 0.0):
    """Move old intraday rows into archive based on MAIN_TABLE_RETENTION_DAYS.

    - Uses UTC epoch seconds in `ts`.
    - Skips daily (`1d`) and any keep_days >= 99999.
    - If batch_limit is set, processes in chunks to reduce long transactions.
    """

    engine = _ensure_tables()
    now_utc = datetime.now(timezone.utc)

    with engine.begin() as conn:
        for interval, keep_days in MAIN_TABLE_RETENTION_DAYS.items():
            if interval == "1d" or keep_days >= 99999:
                continue

            cutoff_ts = int((now_utc - timedelta(days=int(keep_days))).timestamp())

            if batch_limit is None:
                insert_sql = """
                INSERT INTO stock_intraday_archive (symbol, data_type, ts, open, high, low, close, volume)
                SELECT symbol, data_type, ts, open, high, low, close, volume
                FROM stock_ohlc
                WHERE data_type = :interval AND ts < :cutoff_ts
                ON CONFLICT ON CONSTRAINT uq_symbol_datatype_ts_archive DO NOTHING
                """
                conn.execute(text(insert_sql), {"interval": interval, "cutoff_ts": cutoff_ts})

                delete_sql = """
                DELETE FROM stock_ohlc
                WHERE data_type = :interval AND ts < :cutoff_ts
                """
                deleted = conn.execute(text(delete_sql), {"interval": interval, "cutoff_ts": cutoff_ts}).rowcount
                print(f"[ARCHIVE MOVE] interval={interval} deleted_from_main={deleted} cutoff_ts={cutoff_ts}")

            else:
                # Batch mode: move+delete in smaller chunks
                while True:
                    move_sql = """
                    WITH moved AS (
                        SELECT symbol, data_type, ts, open, high, low, close, volume
                        FROM stock_ohlc
                        WHERE data_type = :interval AND ts < :cutoff_ts
                        ORDER BY ts
                        LIMIT :limit
                    ), ins AS (
                        INSERT INTO stock_intraday_archive (symbol, data_type, ts, open, high, low, close, volume)
                        SELECT symbol, data_type, ts, open, high, low, close, volume
                        FROM moved
                        ON CONFLICT ON CONSTRAINT uq_symbol_datatype_ts_archive DO NOTHING
                        RETURNING 1
                    )
                    DELETE FROM stock_ohlc o
                    USING moved m
                    WHERE o.symbol = m.symbol AND o.data_type = m.data_type AND o.ts = m.ts
                    RETURNING 1
                    """
                    res = conn.execute(text(move_sql), {"interval": interval, "cutoff_ts": cutoff_ts, "limit": int(batch_limit)})
                    deleted = res.rowcount
                    if not deleted:
                        break
                    print(f"[ARCHIVE MOVE] interval={interval} deleted_batch={deleted} cutoff_ts={cutoff_ts}")
                    if sleep_sec:
                        time.sleep(float(sleep_sec))


def cleanup_archive_expired(batch_limit: int | None = None, sleep_sec: float = 0.0):
    """Delete very old rows from archive based on ARCHIVE_RETENTION_DAYS."""

    engine = _ensure_tables()
    now_utc = datetime.now(timezone.utc)

    with engine.begin() as conn:
        for interval, keep_days in ARCHIVE_RETENTION_DAYS.items():
            if interval == "1d" or keep_days >= 99999:
                continue

            cutoff_ts = int((now_utc - timedelta(days=int(keep_days))).timestamp())

            if batch_limit is None:
                delete_sql = """
                DELETE FROM stock_intraday_archive
                WHERE data_type = :interval AND ts < :cutoff_ts
                """
                deleted = conn.execute(text(delete_sql), {"interval": interval, "cutoff_ts": cutoff_ts}).rowcount
                print(f"[ARCHIVE CLEANUP] interval={interval} deleted_from_archive={deleted} cutoff_ts={cutoff_ts}")

            else:
                while True:
                    delete_sql = """
                    WITH doomed AS (
                        SELECT id
                        FROM stock_intraday_archive
                        WHERE data_type = :interval AND ts < :cutoff_ts
                        ORDER BY ts
                        LIMIT :limit
                    )
                    DELETE FROM stock_intraday_archive a
                    USING doomed d
                    WHERE a.id = d.id
                    RETURNING 1
                    """
                    res = conn.execute(text(delete_sql), {"interval": interval, "cutoff_ts": cutoff_ts, "limit": int(batch_limit)})
                    deleted = res.rowcount
                    if not deleted:
                        break
                    print(f"[ARCHIVE CLEANUP] interval={interval} deleted_batch={deleted} cutoff_ts={cutoff_ts}")
                    if sleep_sec:
                        time.sleep(float(sleep_sec))


# Examples:
# archive_old_intraday(batch_limit=50000, sleep_sec=0.1)
# cleanup_archive_expired(batch_limit=50000, sleep_sec=0.1)
